In [ ]:
import os
import sys
import typing as tp
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

build_dir = Path(__vsc_ipynb_file__).parent.parent / "build"


sys.path.append(str(build_dir / "pacer" / "gps-source"))
sys.path.append(str(build_dir / "pacer" / "geometry"))

In [ ]:
from _pacer_geometry_impl import (
    CoordinateSystem,
    GPSSample,
    Vec3f,
)
from _pacer_gps_source_impl import (
    DatVersion,
    GPMFSource,
    GPSSource,
    SequentialGPSSource,
    read_dat_file,
)

In [ ]:
def get_dat_df(filename: str) -> pd.DataFrame:
    lats, lons, ts, speeds = [], [], [], []

    read_dat_file(
        filename,
        lambda s, t: (
            lats.append(s.latitude),
            lons.append(s.longitude),
            ts.append(t),
            speeds.append(s.full_speed),
        ),
        DatVersion.WITH_TIMESTAMP,
    )

    return pd.DataFrame(
        {"latitude": lats, "longitude": lons, "timestamp": ts, "speed": speeds}
    )


def display_dat_file(filename: str) -> None:
    df = get_dat_df(filename)
    return px.scatter_map(
        df, lat="latitude", lon="longitude", color="speed", map_style="outdoors"
    ).update_layout(height=800)

In [ ]:
display_dat_file(dat_files[0])

In [ ]:
import pacer._pacer_geometry_impl as pacer_geometry

In [ ]:
display_dat_file(dat_files[1])

In [ ]:
display_dat_file(dat_files[2])

In [ ]:
display_dat_file(dat_files[3])

In [ ]:
s = GPMFSource("data/GH020252.MP4")
points = list(read_all_gps_data(s))

In [ ]:
points[3]

In [ ]:
points_df = pd.DataFrame(
    {
        "latitude": [p.latitude for p, *_ in points],
        "longitude": [p.longitude for p, *_ in points],
        # "timestamp": [p.timestamp for p, *_ in points],
        "speed": [p.full_speed for p, *_ in points],
    }
)

In [ ]:
points_df

In [ ]:
px.scatter_map(
    points_df,
    lat="latitude",
    lon="longitude",
    color="speed",
    map_style="outdoors",
).update_layout(height=800)

In [ ]:
df = get_dat_df(dat_files[3])
df

In [ ]:
df["timestamp"] - df.iloc[0]["timestamp"]

In [ ]:
1702 / 8002

In [ ]:
df = get_dat_df(dat_files[1])
(
    df
    # .loc[
    #     lambda d: (
    #         d[["latitude", "longitude", "speed"]]
    #         == d[["latitude", "longitude", "speed"]].shift(1)
    #     ).any(axis="columns")
    # ]
    .pipe(
        lambda d: px.scatter(
            x=d["timestamp"],
            y=d["timestamp"].diff(),
            log_y=True,
            labels={"x": "timestamp", "y": "diff"},
            title="Time difference between points",
        )
    )
)

In [ ]:
px.histogram(np.log(df["timestamp"].diff()) / np.log(10), nbins=300, labels={"x": "log10(diff)"}, title="Log10 of time difference between points")

In [ ]:
def read_all_gps_data(s: GPSSource) -> tp.Iterable[tuple[GPSSample, float, float]]:
    """
    Read all GPS data from the given GPSSource and return it as a pandas DataFrame.
    """
    gps_data = []
    while not s.IsEnd():
        s.Samples(lambda s, i, n: gps_data.append(s))
        for si in gps_data:
            yield si, *s.CurrentTimeSpan()
        gps_data.clear()
        s.Next()

In [ ]:
s = GPMFSource("data/GH010219.MP4")

In [ ]:
df = pd.DataFrame(
    {
        "latitude": si.latitude,
        "longitude": si.longitude,
        "altitude": si.altitude,
        "3d_speed": si.full_speed * 3.6,
        "2d_speed": si.ground_speed * 3.6,
        "time_in": t0,
        "time_out": t1,
    }
    for si, t0, t1 in read_all_gps_data(s)
)

df

In [ ]:
fig = px.scatter(
    df.loc[lambda d: d["3d_speed"] > 0.1],
    x="longitude",
    y="latitude",
    color="3d_speed",
    color_continuous_scale="Viridis",
    title="GPS Data (Colored by 3D Speed)",
)
fig

In [ ]:
px.line(df["3d_speed"], title="3D Speed over Time")